# Bird Training Pipeline (CUB-200-2011)
EfficientNet-B0 transfer learning notebook with artifact export to backend models directory.

In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
from pathlib import Path

import gdown
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
DATA_DIR = Path('data')
DATASET_DIR = DATA_DIR / 'CUB_200_2011'
ARCHIVE_PATH = DATA_DIR / 'CUB_200_2011.tgz'

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Public mirrors can change; update URL if needed.
CUB_URL = 'https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1'

if not DATASET_DIR.exists():
    if not ARCHIVE_PATH.exists():
        print('Downloading CUB-200-2011...')
        gdown.download(CUB_URL, str(ARCHIVE_PATH), quiet=False, fuzzy=True)

    print('Extracting dataset...')
    import tarfile
    with tarfile.open(ARCHIVE_PATH, 'r:gz') as tar:
        tar.extractall(DATA_DIR)

print('Dataset path:', DATASET_DIR.resolve())

In [ ]:
images_txt = DATASET_DIR / 'images.txt'
labels_txt = DATASET_DIR / 'image_class_labels.txt'
classes_txt = DATASET_DIR / 'classes.txt'
images_root = DATASET_DIR / 'images'

image_id_to_path = {}
with images_txt.open('r', encoding='utf-8') as f:
    for line in f:
        idx, rel_path = line.strip().split(' ', 1)
        image_id_to_path[int(idx)] = rel_path

image_id_to_label = {}
with labels_txt.open('r', encoding='utf-8') as f:
    for line in f:
        idx, cls = line.strip().split(' ', 1)
        image_id_to_label[int(idx)] = int(cls) - 1  # zero-based

class_names = []
with classes_txt.open('r', encoding='utf-8') as f:
    for line in f:
        _, class_name = line.strip().split(' ', 1)
        class_names.append(class_name.replace('_', ' '))

samples = []
for idx, rel_path in image_id_to_path.items():
    label = image_id_to_label[idx]
    samples.append((images_root / rel_path, label))

print('Samples:', len(samples), 'Classes:', len(class_names))

In [ ]:
train_samples, val_samples = train_test_split(
    samples,
    test_size=0.2,
    random_state=SEED,
    stratify=[label for _, label in samples],
)

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BirdDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        path, label = self.rows[idx]
        image = Image.open(path).convert('RGB')
        return self.transform(image), label

train_ds = BirdDataset(train_samples, train_tfms)
val_ds = BirdDataset(val_samples, val_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 200)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

best_acc = 0.0
epochs = 5  # Increase for stronger accuracy

for epoch in range(1, epochs + 1):
    model.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    val_loss, val_acc = evaluate(val_loader)
    print(f'Epoch {epoch:02d} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'bird_model.pth')

print('Best validation accuracy:', best_acc)
target_acc = 0.70
metrics = {'best_val_accuracy': best_acc, 'target_accuracy': target_acc}
Path('training_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
if best_acc < target_acc:
    print(f'⚠️ Validation accuracy {best_acc:.4f} is below target {target_acc:.2f}. Consider more epochs/tuning.')

In [ ]:
Path('label_list.json').write_text(json.dumps(class_names, indent=2), encoding='utf-8')
print('Saved bird_model.pth and label_list.json in training directory')

In [ ]:
backend_models = Path('../apps/backend/models')
backend_models.mkdir(parents=True, exist_ok=True)

shutil.copy2('bird_model.pth', backend_models / 'bird_model.pth')
shutil.copy2('label_list.json', backend_models / 'label_list.json')

print('Artifacts copied to', backend_models.resolve())